In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 74 — Retriever Comparison Harness (Haystack)

## Goal
Swap **retrievers, rerankers, and prompt strategies** under
one evaluation setup to find the best combination.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **BM25 retriever** | Keyword-based (TF-IDF) retrieval |
| **Embedding retriever** | Semantic similarity search |
| **Prompt strategies** | Different prompt templates affect answer quality |
| **Evaluation** | Measure retrieval relevance and answer quality |
| **Comparison harness** | Fair side-by-side testing framework |

## Stack
- `haystack-ai` 2.27+
- `ollama-haystack`
- `sentence-transformers` for embeddings

In [ ]:
import os, warnings, time
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from haystack import Document, Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import (
    InMemoryBM25Retriever,
    InMemoryEmbeddingRetriever,
)
from haystack.components.embedders import (
    SentenceTransformersDocumentEmbedder,
    SentenceTransformersTextEmbedder,
)
from haystack.components.builders import PromptBuilder
from haystack_integrations.components.generators.ollama import OllamaChatGenerator
from haystack.dataclasses import ChatMessage

print("All imports OK")

## Step 1 — Pool of Test Documents & Questions

We create a knowledge base and a set of test queries
with expected relevant documents.

In [ ]:
# Knowledge base documents
raw_docs = [
    {"content": """Gradient Boosting is an ensemble machine learning method that builds models
sequentially, each correcting errors of the previous one. Popular implementations:
XGBoost (regularized), LightGBM (histogram-based, fast), CatBoost (handles categoricals).
Typically outperforms random forests on tabular data. Prone to overfitting without
proper regularization (learning rate, max depth, early stopping).""",
     "meta": {"id": "ml01", "topic": "ML"}},
    
    {"content": """Transformers are deep learning models based on self-attention mechanisms.
Architecture: encoder (BERT-style) or decoder (GPT-style) or both (T5). Key components:
multi-head attention, positional encoding, feed-forward layers, layer normalization.
Applications: NLP (translation, summarization), vision (ViT), audio (Whisper).
Pre-training + fine-tuning is the dominant paradigm.""",
     "meta": {"id": "ml02", "topic": "ML"}},
    
    {"content": """Retrieval-Augmented Generation (RAG) enhances LLMs by fetching relevant
documents before generating answers. Pipeline: query → retriever → context + query → LLM.
Benefits: reduces hallucination, enables knowledge updates without retraining.
Challenges: retrieval quality, chunk size optimization, context window limits.
Advanced: HyDE (hypothetical document embeddings), query rewriting, reranking.""",
     "meta": {"id": "ml03", "topic": "ML"}},
    
    {"content": """JWT (JSON Web Token) is a compact, URL-safe token format for authentication.
Structure: header.payload.signature (base64url encoded). The header specifies the
algorithm (HS256, RS256). The payload contains claims (sub, exp, iat, custom).
Stateless: server doesn't store sessions. Risks: token theft, no revocation without
blocklists, payload is readable (not encrypted by default).""",
     "meta": {"id": "sec01", "topic": "Security"}},
    
    {"content": """OAuth 2.0 is an authorization framework that enables third-party access
to resources without sharing credentials. Grant types: Authorization Code (web apps),
Client Credentials (service-to-service), PKCE (mobile/SPA). Key concepts: access tokens,
refresh tokens, scopes, consent screen. OpenID Connect (OIDC) adds authentication on top.""",
     "meta": {"id": "sec02", "topic": "Security"}},
    
    {"content": """GraphQL is an API query language that lets clients request exactly the data
they need. Advantages over REST: no over/under-fetching, single endpoint, strong typing
via schema. Schema defines types, queries, mutations, subscriptions. Tools: Apollo Server,
Hasura, Strawberry (Python). Challenges: N+1 queries, caching complexity, file uploads.""",
     "meta": {"id": "api01", "topic": "API"}},
]

# Test queries with expected relevant doc IDs
test_queries = [
    {"query": "How does gradient boosting work and what are popular implementations?",
     "expected": ["ml01"]},
    {"query": "Explain the transformer architecture and self-attention",
     "expected": ["ml02"]},
    {"query": "What is RAG and how does it reduce hallucination?",
     "expected": ["ml03"]},
    {"query": "How do JWT tokens work for authentication?",
     "expected": ["sec01"]},
    {"query": "Compare OAuth 2.0 grant types for web and mobile apps",
     "expected": ["sec02"]},
    {"query": "What are the advantages of GraphQL over REST APIs?",
     "expected": ["api01"]},
]

print(f"Knowledge base: {len(raw_docs)} documents")
print(f"Test queries: {len(test_queries)}")

## Step 2 — Setup BM25 and Embedding Retrievers

In [ ]:
# --- BM25 Store ---
bm25_store = InMemoryDocumentStore()
bm25_docs = [Document(content=d["content"], meta=d["meta"]) for d in raw_docs]
bm25_store.write_documents(bm25_docs)

bm25_retriever = InMemoryBM25Retriever(document_store=bm25_store, top_k=3)
print(f"BM25 store: {bm25_store.count_documents()} docs")

# --- Embedding Store ---
embed_store = InMemoryDocumentStore()
embed_docs = [Document(content=d["content"], meta=d["meta"]) for d in raw_docs]

# Embed documents
doc_embedder = SentenceTransformersDocumentEmbedder(
    model="BAAI/bge-small-en-v1.5"
)
doc_embedder.warm_up()
embedded = doc_embedder.run(documents=embed_docs)
embed_store.write_documents(embedded["documents"])

# Text embedder for queries 
text_embedder = SentenceTransformersTextEmbedder(
    model="BAAI/bge-small-en-v1.5"
)
text_embedder.warm_up()

embed_retriever = InMemoryEmbeddingRetriever(document_store=embed_store, top_k=3)
print(f"Embedding store: {embed_store.count_documents()} docs (embedded)")

## Step 3 — Define Prompt Strategies

In [ ]:
# Strategy 1: Simple prompt
simple_template = """
Answer the question based on the context.

Context:
{% for doc in documents %}
{{ doc.content }}
{% endfor %}

Question: {{ question }}
Answer:"""

# Strategy 2: Structured prompt with instructions
structured_template = """
You are a technical expert. Use ONLY the provided context to answer.
If the context is insufficient, say "I don't have enough information."

Rules:
1. Be concise but complete
2. Cite specific technologies mentioned in the context
3. Use bullet points for lists

Context:
{% for doc in documents %}
--- Document [{{ doc.meta.id }}] ---
{{ doc.content }}
{% endfor %}

Question: {{ question }}

Structured Answer:"""

# Strategy 3: Chain-of-thought prompt
cot_template = """
You are a technical expert. Think step by step.

Context:
{% for doc in documents %}
{{ doc.content }}
{% endfor %}

Question: {{ question }}

Step 1: Identify which context documents are relevant.
Step 2: Extract the key facts.
Step 3: Compose a precise answer.

Answer:"""

prompt_strategies = {
    "simple": PromptBuilder(template=simple_template),
    "structured": PromptBuilder(template=structured_template),
    "chain_of_thought": PromptBuilder(template=cot_template),
}

print(f"Prompt strategies: {list(prompt_strategies.keys())}")

## Step 4 — Evaluation Harness

In [ ]:
generator = OllamaChatGenerator(
    model="qwen3.5:9b",
    url="http://localhost:11434",
)

def evaluate_retriever(retriever_name, retrieve_fn, prompt_name, prompt_builder):
    """Evaluate a retriever + prompt combination."""
    results = []
    
    for tq in test_queries:
        query = tq["query"]
        expected = set(tq["expected"])
        
        # Retrieve
        t0 = time.time()
        docs = retrieve_fn(query)
        retrieval_time = time.time() - t0
        
        # Check retrieval accuracy (hit@k)
        retrieved_ids = {d.meta["id"] for d in docs}
        hits = len(expected & retrieved_ids)
        recall = hits / len(expected) if expected else 0
        
        # Generate answer
        prompt_result = prompt_builder.run(documents=docs, question=query)
        messages = [ChatMessage.from_user(prompt_result["prompt"])]
        t0 = time.time()
        gen_result = generator.run(messages=messages)
        gen_time = time.time() - t0
        answer = gen_result["replies"][0].text
        
        results.append({
            "query": query[:50],
            "recall": recall,
            "retrieval_ms": retrieval_time * 1000,
            "generation_s": gen_time,
            "answer_len": len(answer),
            "retrieved": list(retrieved_ids),
        })
    
    avg_recall = sum(r["recall"] for r in results) / len(results)
    avg_ret_ms = sum(r["retrieval_ms"] for r in results) / len(results)
    avg_gen_s = sum(r["generation_s"] for r in results) / len(results)
    
    return {
        "retriever": retriever_name,
        "prompt": prompt_name,
        "avg_recall": avg_recall,
        "avg_retrieval_ms": avg_ret_ms,
        "avg_generation_s": avg_gen_s,
        "details": results,
    }

print("Evaluation harness ready")

In [ ]:
# Define retriever functions
def bm25_retrieve(query):
    return bm25_retriever.run(query=query)["documents"]

def embedding_retrieve(query):
    emb = text_embedder.run(text=query)
    return embed_retriever.run(query_embedding=emb["embedding"])["documents"]

# Run comparisons (BM25 + simple, BM25 + structured, Embedding + simple, Embedding + structured)
configs = [
    ("BM25", bm25_retrieve, "simple"),
    ("BM25", bm25_retrieve, "structured"),
    ("Embedding", embedding_retrieve, "simple"),
    ("Embedding", embedding_retrieve, "structured"),
]

all_results = []
for ret_name, ret_fn, prompt_name in configs:
    print(f"\nEvaluating: {ret_name} + {prompt_name}...")
    result = evaluate_retriever(ret_name, ret_fn, prompt_name, prompt_strategies[prompt_name])
    all_results.append(result)
    print(f"  Recall: {result['avg_recall']:.2%} | Retrieval: {result['avg_retrieval_ms']:.1f}ms | Gen: {result['avg_generation_s']:.1f}s")

In [ ]:
# Summary table
print(f"\n{'='*75}")
print(f"{'Retriever':<12} {'Prompt':<16} {'Recall':>8} {'Ret(ms)':>10} {'Gen(s)':>8}")
print(f"{'='*75}")

for r in all_results:
    print(f"{r['retriever']:<12} {r['prompt']:<16} {r['avg_recall']:>7.0%} {r['avg_retrieval_ms']:>9.1f} {r['avg_generation_s']:>7.1f}")

# Find best config
best = max(all_results, key=lambda x: x["avg_recall"])
print(f"\n🏆 Best config: {best['retriever']} + {best['prompt']} (Recall: {best['avg_recall']:.0%})")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **BM25 Retriever** | Keyword matching with TF-IDF scoring |
| **Embedding Retriever** | Cosine similarity on dense vectors |
| **Prompt strategies** | Simple vs structured vs chain-of-thought |
| **Recall@K** | Fraction of expected docs in top-K results |
| **Comparison harness** | Fair evaluation across configurations |

## Retriever Trade-offs
```
                Exact Match    Semantic Match    Speed
BM25            ★★★★★          ★★               ★★★★★
Embedding       ★★             ★★★★★            ★★★
Hybrid          ★★★★           ★★★★             ★★★
```

## Extending This Notebook
- Add a **Hybrid retriever** (BM25 + Embedding with reciprocal rank fusion)
- Add a **Reranker** (cross-encoder) after retrieval
- Test with more embedding models (e.g., `all-MiniLM-L6-v2`)
- Add **LLM-as-judge** evaluation for answer quality
- Increase document count for more realistic benchmarks